# Module 4: Dense versus MoE on the GPU

In Module 3 you proved one request leaves the GPU memory-bound: to make one token, the server reads every weight once. This module gives that fact a picture and a name, the roofline, then uses it to explain why a mixture-of-experts model generates faster than its size suggests. This is the first "read fewer bytes" idea, and it hands off to Omer's optimization block.

## Learning objectives
- Plot compute against memory bandwidth and read the ridge point for your card
- Place decode and prefill on that plot and say which is memory-bound
- Explain why decode reads every weight to make one token
- Derive active versus total parameters for a mixture-of-experts model from its real config
- Measure the reasoning tax of a thinking model
- Name the ways to read fewer bytes, and which module covers each

## Prerequisites
- Finished Module 3
- Your vLLM Deployment serves `Qwen/Qwen3-4B` for the two live measurements
- Outbound access to the Hugging Face Hub for the MoE config, or use the bundled copy
- About 15 minutes

References: [vLLM](https://docs.vllm.ai) &middot; [Anatomy of vLLM](https://blog.vllm.ai/2025/09/05/anatomy-of-vllm.html) &middot; [Qwen3-30B-A3B model card](https://huggingface.co/Qwen/Qwen3-30B-A3B) &middot; [Roofline model](https://en.wikipedia.org/wiki/Roofline_model) &middot; [Akamai Cloud GPUs](https://www.akamai.com/products/gpu)

## Memory-bound design basics

Decode is memory-bound. To make one token, the server reads every weight in the model once, so the token rate is capped by how fast the card reads memory. Every technique in the optimization block is a way to read fewer bytes or reuse the bytes you read.

- Arithmetic intensity is the work done per byte read, in FLOP per byte.
- Plot two limits against it: a sloped memory-bandwidth line and a flat compute line. Where they cross is the ridge point.
- Decode at batch 1 reads every weight to make one token, about one FLOP per byte, so it sits far down the memory side. Prefill runs many tokens in one pass and reuses each loaded weight across all of them, many FLOP per byte, so it sits near the compute line. Same card, two different places.

![The memory-bandwidth line and the compute line meeting at the ridge point, with decode far down the memory side and prefill near the top](images/04_dense_vs_moe_architecture.png)

## 1. Setup

The roofline plot and the MoE derivation need only numpy and matplotlib, plus the bundled config under `assets/`. The two live measurements in sections 5 and 6 use your client.

In [ ]:
%pip install -q "openai>=1.40" "requests>=2.31" "numpy>=1.26" "matplotlib>=3.8"

In [ ]:
import sys, time
from pathlib import Path

if Path("../common").exists():
    REPO_ROOT = Path("..")
else:
    REPO_ROOT = Path(".")
sys.path.insert(0, str(REPO_ROOT.resolve()))

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from common.config import get_settings, print_settings, build_client
from common import foundation

settings = get_settings()
settings.model_name = foundation.served_model_name(settings)  # use the model the server actually serves
print_settings(settings)
client = build_client(settings)
ASSETS = REPO_ROOT / "04_dense_vs_moe" / "assets"
MOE = "Qwen/Qwen3-30B-A3B"

**What you should see:** your settings. The roofline and the MoE derivation run with no GPU; the two measurements need your endpoint.

## 2. Memory-bound against compute-bound: the roofline

Plot the two limits. The compute line is flat at the card's peak FLOP per second. The memory line slopes up with arithmetic intensity, at the card's bandwidth. Below the ridge you are memory-bound, above it compute-bound. Use the RTX 4000 Ada numbers: about 107 TFLOP/s dense FP16 and 360 GB/s. Use the dense figure, not the 427 on the spec sheet, which is FP8 with 2:1 sparsity.

In [ ]:
# The roofline: where a job flips from waiting on memory to waiting on compute.
peak_tflops = 107.0       # RTX 4000 Ada, dense FP16
bandwidth_tbs = 0.36      # 360 GB/s
ridge = peak_tflops / bandwidth_tbs
print(f"ridge point: {ridge:.0f} FLOP/byte  (below this you are memory-bound)")

intensity = np.logspace(-1, 3, 200)
attainable = np.minimum(peak_tflops, bandwidth_tbs * intensity)

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.fill_between(intensity, 1e-2, attainable, where=intensity < ridge, color="#d9534f", alpha=0.10)
ax.fill_between(intensity, 1e-2, attainable, where=intensity >= ridge, color="#2fa56a", alpha=0.10)
ax.plot(intensity, attainable, color="#1b1a3d", lw=2)
ax.axvline(ridge, ls="--", color="#9a9ab0")
ax.scatter([1.0], [bandwidth_tbs], color="#d9534f", zorder=5, label="decode at batch 1 (memory-bound)")
ax.scatter([ridge * 3], [peak_tflops], color="#2fa56a", zorder=5, label="prefill (compute-bound)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("arithmetic intensity (FLOP/byte)")
ax.set_ylabel("attainable TFLOP/s")
ax.set_title(f"Memory-bound below {ridge:.0f} FLOP/byte, compute-bound above")
ax.legend(); ax.grid(True, which="both", alpha=0.3); fig.tight_layout()
fig.savefig(REPO_ROOT / "04_dense_vs_moe" / "images" / "04_roofline_live.png", dpi=130)
plt.close(fig)
print(f"decode at intensity ~1 reaches {bandwidth_tbs:.2f} TFLOP/s, about {bandwidth_tbs / peak_tflops * 100:.2f}% of peak")

**What you should see:** a ridge near 297 FLOP per byte, a red dot for decode far down the sloped memory side, and a tiny fraction of peak compute reached. The tensor cores wait on memory. You measured this split in Module 3; this is the picture of it.

## 3. MoE: the first way to read fewer weights

A dense model reads all its weights per token. A mixture-of-experts model holds many experts but routes each token to only a few, so it reads only the active ones. Total parameters set the memory footprint. Active parameters set the bandwidth cost, and bandwidth caps decode. Derive both from the real config of `Qwen3-30B-A3B` rather than taking the numbers on faith.

In [ ]:
# Derive active vs total parameters from the real MoE config.
cfg = foundation.load_model_config(MOE, fallback_dir=str(ASSETS))
total = sum(foundation.model_param_count(MOE, fallback_dir=str(ASSETS)).values())

L = cfg["num_hidden_layers"]
H = cfg["hidden_size"]
moe_inter = cfg["moe_intermediate_size"]
E = cfg["num_experts"]
k = cfg["num_experts_per_tok"]

# Each expert MLP is gate_proj + up_proj + down_proj = 3 x hidden x moe_intermediate.
expert_params = E * 3 * H * moe_inter * L
shared_params = total - expert_params               # attention, embeddings, router, norms
active = shared_params + (k / E) * expert_params     # only k of E experts run per token

print(f"{MOE}: {total/1e9:.1f}B total parameters")
print(f"  {E} experts, {k} active per token, {L} layers, hidden {H}, expert MLP width {moe_inter}")
print(f"  expert weights : {expert_params/1e9:.1f}B  (stored, but only {k}/{E} run per token)")
print(f"  shared weights : {shared_params/1e9:.2f}B  (run every token)")
print(f"  ACTIVE / token : {active/1e9:.2f}B  -> this is the 'A3B' in the name")

**What you should see:** about 30.5B total parameters but only about 3.3B active per token, derived from the config: most of the weight is in the 128 experts, and only 8 of them run for any given token. The active count is what decode reads.

In [ ]:
# Bytes moved per token. The MoE wins by reading only its active experts.
bpp = 2  # BF16
dense4_read = 4.0e9 * bpp
dense30_read = total * bpp                 # a dense 30B would read all of it
moe_read = active * bpp                    # the MoE reads only the active params
moe_store = total * bpp                    # but stores every expert

print(f"dense 4B    : {dense4_read/1e9:5.1f} GB read per token")
print(f"dense 30B   : {dense30_read/1e9:5.1f} GB read per token  (hypothetical, for contrast)")
print(f"MoE 30B-A3B : {moe_read/1e9:5.1f} GB read per token, but {moe_store/1e9:.0f} GB stored")
print(f"=> the MoE reads {dense30_read/moe_read:.1f}x fewer bytes than a dense 30B, "
      f"so it decodes about that much faster")

**What you should see:** the MoE reads about 6.7 GB per token, roughly nine times less than a dense 30B, so it decodes about nine times faster than a dense 30B. You pay the per-token bandwidth of a small model and the VRAM footprint of a large one; quality lands between its active and total size, well above a 3B dense model but short of the full 30B. The catch is footprint: the whole 30.5B, all 128 experts plus the shared weights, sits in VRAM, about 61 GB at BF16, 30 GB at FP8, 15 to 18 GB at INT4. Even the FP8 footprint overflows the 20 GB card, so this stays a derivation, not a live run. The multi-GPU lever that fits an MoE is expert parallelism, in the Things to know below.

## 4. Compare dense sizes without switching the server

Keep the server on the 4B target model. The 0.6B model is pre-cached for Module 6 as a speculative decoding draft model, so do not switch vLLM to serve it directly here. Instead measure the decode rate on the 4B you are serving and compare it with the bandwidth ceiling for a smaller model. The lesson is the same: fewer weight bytes per token means a higher memory-bound ceiling.

In [ ]:
# Requires a live vLLM endpoint. Stream one answer and time the decode.
def decode_rate(model):
    start = time.time(); first = None; n = 0
    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "Write a detailed paragraph about GPU memory."}],
        max_tokens=200, temperature=0.0, stream=True,
    )
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            if first is None:
                first = time.time()
            n += 1
    secs = (time.time() - first) if first else 0.0
    return n / secs if secs else 0.0

bandwidth_gb_s = 360
served_gb = 4.0e9 * 2 / 1e9      # 4B at BF16
measured = decode_rate(settings.model_name)
served_ceiling = bandwidth_gb_s / served_gb
small_ceiling = bandwidth_gb_s / (0.6e9 * 2 / 1e9)   # 0.6B at BF16, same precision as the 4B, so size is the only variable

print(f"{settings.model_name}: measured about {measured:.0f} tok/s")
print(f"{settings.model_name}: bandwidth ceiling about {served_ceiling:.0f} tok/s at BF16")
print(f"0.6B model            : ceiling about {small_ceiling:.0f} tok/s at BF16 (6.7x fewer weight bytes)")
print("Do not serve the 0.6B directly here; Module 6 uses it beside the 4B target.")

**What you should see:** a measured rate below but near the bandwidth-over-weights ceiling from Module 2. Real decode also reads the KV cache and pays kernel overhead, so expect somewhat less than the round number. A 0.6B model would clear far more tokens per second because it reads far fewer weight bytes; that 0.6B becomes the draft model for speculative decoding in Module 6, where the 4B still verifies the output.

## 5. The reasoning tax

Decode latency is two things multiplied: how many tokens you generate times the time per token. Sections 2 through 4 attacked the time per token, the bytes read. The reasoning trace attacks the other factor. Qwen3 models are thinking models: by default they write a reasoning trace before the answer, which helps tool choice and hurts latency. The measurement cells in this workshop turn thinking off (`build_client` does it) so your numbers are the answer, not the trace. Measure the cost when it is on.

In [ ]:
# Requires a live vLLM endpoint. Same question, thinking off (the measurement default) vs on.
def timed(enable_thinking):
    t0 = time.time()
    r = client.chat.completions.create(
        model=settings.model_name,
        messages=[{"role": "user", "content": "Is a 4B or a 0.6B cheaper to run, and why? One short answer."}],
        max_tokens=600, temperature=0.0,
        extra_body={"chat_template_kwargs": {"enable_thinking": enable_thinking}},
    )
    return r.usage.completion_tokens, time.time() - t0

off_tok, off_s = timed(False)
on_tok, on_s = timed(True)
rate = off_tok / off_s if off_s else 60
print(f"thinking OFF: {off_tok:>4} tokens in {off_s:.1f}s")
print(f"thinking ON : {on_tok:>4} tokens in {on_s:.1f}s  (+{on_tok - off_tok} reasoning tokens)")
print(f"at ~{rate:.0f} tok/s those extra tokens cost about {(on_tok - off_tok) / rate:.0f}s per agent step")

**What you should see:** thinking off returns a short answer fast. Thinking on emits a long trace first, often hundreds of tokens, so it takes several times longer. Multiply by every step of an agent loop and reasoning is a real latency budget. On a fixed context cap a runaway trace can even fill the output and return no answer. The lever: thinking off for steps that just need a tool call, on for steps that need a plan.

## 6. What this means for owning your inference

The memory-bound limit is how you pick a model for a card. A small dense model and a sparse MoE can hold the same latency on the same GPU for different reasons: the dense model is small enough to read fast, and the MoE is large but reads only its active experts. Knowing which you have tells you what to expect and what to tune.

Each layer you have met multiplies the others. This is the map to leave with.

| Layer | What it wins | Mechanism | For your agent |
|---|---|---|---|
| Algorithm | skip recompute | the KV cache (Module 3) | every decode step is cheap |
| Reuse | skip re-prefill | prefix caching (Module 3) | the resent system prompt is free |
| Architecture | a smaller cache | GQA, fewer KV heads (Module 2) | more agents fit per card |
| Precision | fewer bytes per weight | FP8 quantization (Module 5) | faster decode, more KV room |
| System | share the weight read | continuous batching (Module 7) | one card serves a fleet of agents |

## 7. Look forward: split prefill from decode

You proved prefill is compute-bound and decode is memory-bound. On one GPU they fight: a big prefill stalls everyone's decode. The next move in production serving is to stop sharing. Put prefill on one pool of GPUs and decode on another, and ship the KV cache between them over a fast link. That is disaggregated serving (vLLM's `--kv-transfer-config` with a NIXL connector, NVIDIA Dynamo, llm-d). You cannot run it on one 20 GB card, so this is the look-forward, not a lab. The win is narrow. A single prefill-plus-decode pair does not raise throughput by itself. The gains come at fleet scale with smart routing, where you size and scale the two phases independently and stop a prefill spike from ever interrupting a decode.

![A prefill GPU builds the KV cache and ships it over NIXL to a decode GPU](images/04_disaggregation.png)

In [ ]:
# Not runnable on one GPU. The idea, and the flags for the day you have two.
print("On one GPU, a big prefill and an active decode share the same engine, so the prefill")
print("spike stalls the decode. Splitting them onto separate GPUs removes the interference.\n")
print("The flags, the day you have two GPUs:")
print('  prefill GPU:  --kv-transfer-config \'{"kv_connector":"NixlConnector","kv_role":"kv_producer"}\'')
print('  decode  GPU:  --kv-transfer-config \'{"kv_connector":"NixlConnector","kv_role":"kv_consumer"}\'')

## 8. Hand off to Omer

Decode is memory-bound, and you just saw the first way to beat it: an MoE reads only its active experts. The optimization block is the rest of that ladder, each lever a way to read fewer bytes or reuse the ones you read: quantization (Module 5), speculative decoding (Module 6), and the engine's attention kernels and continuous batching (Module 7). Each points back to the roofline you drew.

## Things to know

- **The limit is per operation.** Prefill sits near the compute line, decode far down the memory side, on the same card.
- **The ridge clusters near 300 FLOP/byte.** Put Akamai's RTX PRO 6000 Blackwell numbers in (about 500 TFLOP/s dense FP16, 1597 GB/s) and the ridge stays near 313. Compute and bandwidth scale together, so decode is memory-bound on every card you can rent.
- **MoE trades footprint for bandwidth.** You store every expert and read only the active ones. You pay in VRAM to save on the per-token read.
- **Use the dense figure for the card peak.** The RTX 4000 Ada is about 107 TFLOP/s dense FP16. The 427 on the spec sheet is FP8 with 2:1 sparsity.
- **Expert parallelism is the MoE-only multi-GPU lever.** Dense models scale with tensor and pipeline parallelism.

**Recompute the ridge.** Put Akamai's RTX PRO 6000 Blackwell numbers in section 2 (about 500 TFLOP/s dense FP16, 1597 GB/s) and read the ridge. It barely moves, near 313, even though that card has far more compute than the Ada. Compute and memory bandwidth scale together, so the ridge clusters near 300 FLOP/byte across cards, which is why decode stays memory-bound on every one. **Stretch:** mark where decode at batch 16 sits and watch it climb the memory line toward the ridge.

**Compare another MoE.** With `huggingface_hub` installed and Hub access, point `MOE` in section 3 at a different mixture-of-experts model and derive its active versus total parameters and bytes per token.

In [ ]:
# Change these, then run the cell.
your_peak_tflops = 500.0    # Akamai's NVIDIA RTX PRO 6000 Blackwell, dense FP16
your_bw_tbs = 1.597         # 1597 GB/s GDDR7
print(f"ridge point: {your_peak_tflops / your_bw_tbs:.0f} FLOP/byte")

## Summary

- Decode is memory-bound: one token reads every weight once, so memory bandwidth caps the rate.
- The roofline puts compute against memory, and the ridge point is where they cross. Decode sits far down the memory side.
- A mixture-of-experts model reads only its active experts, which you derived from the real config: about 3.3B active of 30.5B total.
- An MoE trades VRAM footprint for a smaller per-token read. It is the first of several ways to beat the memory limit.
- You measured the reasoning tax, the decode time a thinking trace adds to every agent step.

## Next

**Hand off to Omer, Module 5: Quantization.** Decode is memory-bound, and quantization is the next way to read fewer bytes. Omer turns that into a decision: measure the BF16 baseline, switch the deployment to FP8, and compare. You sized FP8 in Module 2; Omer serves it and measures what it costs in quality.